In [ ]:
#@title 1. Connect to Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
save_dir = '/content/drive/MyDrive/NPL2025_Proj'
TRAIN_FILE = os.path.join(save_dir, "train_split.jsonl")
TEST_FILE = os.path.join(save_dir, "val_split.jsonl")

Mounted at /content/drive


In [ ]:
# 1. Uninstall the conflicting library
!pip uninstall -y peft

# 2. Re-run your install
!pip install -q numpy==1.26.4 transformers==4.41.2 datasets==2.19.1 accelerate==0.31.0 torch==2.3.1 scikit-learn==1.6.0 pandas

In [ ]:
#@title 2. Environment + Install Dependencies

import json
import sys
import os
import glob
import zipfile
from collections import defaultdict
from typing import List, Dict, Any
import io

import numpy as np
from datasets import Dataset
from transformers import (
    RobertaTokenizerFast,
    RobertaForTokenClassification,
    Trainer,
    DataCollatorForTokenClassification,
    TrainingArguments,
)

import pandas as pd
from typing import List, Dict, Tuple, Optional
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from transformers import AutoTokenizer

In [ ]:
#@title 3. Configuration and Helper Functions

# Main Config
MODEL_NAME = "roberta-base"
OUTPUT_DIR_BASE = "roberta-single-type-simplified"
MARKER_TYPES = ["Action", "Actor", "Effect", "Evidence", "Victim"]
TEMP_SUBMISSION_FILE = "submission.jsonl"

FINAL_SUBMISSION_ZIP = os.path.join(save_dir, "submission2.zip")

# Training Hyperparameters
TRAIN_BATCH_SIZE = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS = 10

# Inference Hyperparameters
INFER_BATCH_SIZE = 64

# Helper Functions
def load_data(file_path):
    data = []
    with open(file_path, 'r') as f:
        for line in f:
            try:
                item = json.loads(line.strip())
                item["_id"] = item.get("_id", f"sample_{len(data)}")
                item["text"] = item.get("text", "")
                item["markers"] = item.get("markers", [])
                item["conspiracy"] = item.get("conspiracy", "No")
                data.append(item)
            except json.JSONDecodeError:
                print(f"Skipping invalid JSON line: {line.strip()}")
    return data

def create_label_maps_simplified(marker_type):
    label_list = ["O", marker_type]
    label_to_id = {label: i for i, label in enumerate(label_list)}
    id_to_label = {i: label for label, i in label_to_id.items()}
    return label_to_id, id_to_label, len(label_list)

def tokenize_and_align_labels_for_training(examples, tokenizer, label_to_id, marker_type):
    tokenized_inputs = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128, return_offsets_mapping=True)
    labels = []
    all_markers = examples.get("markers", [])
    for i, offsets in enumerate(tokenized_inputs["offset_mapping"]):
        example_labels = [0] * len(offsets)
        example_markers = all_markers[i] if i < len(all_markers) else []
        for marker in example_markers:
            if marker["type"] == marker_type:
                start_char, end_char = marker["startIndex"], marker["endIndex"]
                marker_label = label_to_id.get(marker_type)
                if marker_label is not None:
                    for token_idx, (start, end) in enumerate(offsets):
                        if start is not None and end is not None:
                            if start_char <= start < end_char or (start < end_char and end > start_char):
                                if token_idx < len(example_labels):
                                    example_labels[token_idx] = marker_label
        labels.append(example_labels)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

def tokenize_for_inference(examples, tokenizer):
    tokenized_inputs = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128, return_offsets_mapping=True)
    tokenized_inputs["labels"] = [[-100] * len(offset_map) for offset_map in tokenized_inputs["offset_mapping"]]
    return tokenized_inputs

def find_latest_checkpoint(base_path, marker_type):
    full_path = f"{base_path}-{marker_type}"
    checkpoint_dirs = glob.glob(os.path.join(full_path, "checkpoint-*"))
    if not checkpoint_dirs: return full_path
    checkpoint_dirs.sort(key=lambda x: int(os.path.basename(x).split('-')[-1]))
    return checkpoint_dirs[-1]

def reconstruct_spans(predictions, tokenized_dataset, id_to_label):
    reconstructed_markers = defaultdict(list)
    positive_label_type = id_to_label.get(1)
    if not positive_label_type or positive_label_type == "O": return reconstructed_markers
    for i, pred_ids in enumerate(predictions):
        offsets = tokenized_dataset[i]['offset_mapping']
        original_text = tokenized_dataset[i]['text']
        current_span_start_char = None
        for token_idx, label_id in enumerate(pred_ids):
            offset_tuple = offsets[token_idx]
            is_special = not offset_tuple or offset_tuple[0] == offset_tuple[1]
            if current_span_start_char is not None and (is_special or id_to_label[label_id] == 'O'):
                prev_end_char = offsets[token_idx - 1][1]
                span_text = original_text[current_span_start_char:prev_end_char]
                reconstructed_markers[i].append({"startIndex": current_span_start_char, "endIndex": prev_end_char, "type": positive_label_type, "text": span_text})
                current_span_start_char = None
            if current_span_start_char is None and not is_special and id_to_label[label_id] == positive_label_type:
                current_span_start_char = offset_tuple[0]
        if current_span_start_char is not None:
            last_valid_end = [o[1] for o in offsets if o and o[1] is not None][-1]
            span_text = original_text[current_span_start_char:last_valid_end]
            reconstructed_markers[i].append({"startIndex": current_span_start_char, "endIndex": last_valid_end, "type": positive_label_type, "text": span_text})
    return reconstructed_markers

def save_and_zip(file_path: str, data: List[Dict], output_zip_path: str):
    with open(file_path, 'w', encoding='utf-8') as f:
        for item in data: f.write(json.dumps(item) + '\n')
    with zipfile.ZipFile(output_zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write(file_path, arcname=os.path.basename(file_path))
    os.remove(file_path)
    print(f"Successfully created final submission file and saved to Google Drive: {output_zip_path}")

def compute_final_metrics(ground_truth_data: List[Dict], predicted_data: List[Dict]):
    """
    Computes Exact Match Precision, Recall, and F1-score for the final marker lists
    by comparing predicted_data (from submission) with ground_truth_data (from TEST_FILE).

    This is an exact-match evaluation, which is the standard NER metric.
    """

    gt_map = {item['_id']: item.get('markers', []) for item in ground_truth_data}
    pred_map = {item['_id']: item.get('markers', []) for item in predicted_data}

    results = {}

    for marker_type in MARKER_TYPES:
        TP = 0
        FP = 0
        FN = 0

        for doc_id, gt_markers in gt_map.items():
            pred_markers = pred_map.get(doc_id, [])

            # Filter markers by the current type
            gt_t = set([(m['startIndex'], m['endIndex'], m['type']) for m in gt_markers if m['type'] == marker_type])
            pred_t = set([(m['startIndex'], m['endIndex'], m['type']) for m in pred_markers if m['type'] == marker_type])

            TP += len(gt_t.intersection(pred_t))
            FP += len(pred_t.difference(gt_t))
            FN += len(gt_t.difference(pred_t))

        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        results[marker_type] = {
            'TP': TP, 'FP': FP, 'FN': FN,
            'Precision': round(precision, 4),
            'Recall': round(recall, 4),
            'F1_Score': round(f1, 4)
        }

    # Compute Macro-F1 (average of individual F1s)
    macro_f1 = sum(r['F1_Score'] for r in results.values()) / len(MARKER_TYPES)

    return results, macro_f1

In [ ]:
#@title 4. Train the RoBERTa Models

# Import RoBERTa-specific classes
from transformers import RobertaTokenizerFast, RobertaForTokenClassification

# Load data
train_data = load_data(TRAIN_FILE)
train_dataset = Dataset.from_list(train_data)

tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME, add_prefix_space=True)

for marker_type in MARKER_TYPES:
    print(f"\n--- Training model for marker type: {marker_type} ---")

    label_to_id, id_to_label, num_labels = create_label_maps_simplified(marker_type)

    tokenized_train_dataset = train_dataset.map(
        tokenize_and_align_labels_for_training,
        batched=True,
        fn_kwargs={"tokenizer": tokenizer, "label_to_id": label_to_id, "marker_type": marker_type}
    )

    model = RobertaForTokenClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

    # Define training arguments
    output_dir = f"{OUTPUT_DIR_BASE}-{marker_type}"
    training_args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=0.01,
        logging_steps=len(tokenized_train_dataset) // TRAIN_BATCH_SIZE,
        report_to="none",
        save_strategy="epoch",
        load_best_model_at_end=False,
    )

    data_collator = DataCollatorForTokenClassification(tokenizer)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train_dataset,
        data_collator=data_collator,
        tokenizer=tokenizer,
    )

    # Train the model
    trainer.train()
    print(f"Training for {marker_type} finished.")

print("\n success")

In [ ]:
#@title 5. Inference and Generate Submission File

# Load test data
raw_data = load_data(TEST_FILE)
if not raw_data:
    print("Error: No test data loaded. Cannot perform inference.")
else:
    unique_ids = [d["_id"] for d in raw_data]
    conspiracy_keys = [d["conspiracy"] for d in raw_data]
    test_dataset = Dataset.from_list(raw_data)

    # Tokenize test data
    tokenized_test_dataset = test_dataset.map(
        tokenize_for_inference,
        batched=True,
        remove_columns=[col for col in test_dataset.column_names if col not in ['text', 'offset_mapping', '_id', 'conspiracy']],
        fn_kwargs={"tokenizer": tokenizer}
    )

    all_predicted_markers = defaultdict(list)

    # Iterate and infer for each marker type
    for marker_type in MARKER_TYPES:
        print(f"\n--- Running inference for type: {marker_type} ---")
        model_directory = find_latest_checkpoint(OUTPUT_DIR_BASE, marker_type)

        try:
            # Load the RoBERTa model
            model = RobertaForTokenClassification.from_pretrained(model_directory)
            id_to_label = {0: "O", 1: marker_type}
        except Exception as e:
            print(f"Error loading model for {marker_type}. Details: {e}")
            continue

        # Prepare for inference
        prediction_args = Trainer(
            model=model,
            args=TrainingArguments(output_dir=f"./tmp_inference", per_device_eval_batch_size=INFER_BATCH_SIZE, report_to="none"),
            data_collator=DataCollatorForTokenClassification(tokenizer),
            tokenizer=tokenizer
        )

        # Perform inference
        predictions_output = prediction_args.predict(tokenized_test_dataset)
        predicted_class_ids = np.argmax(predictions_output.predictions, axis=2)

        # Reconstruct and aggregate spans
        current_marker_map = reconstruct_spans(predicted_class_ids, tokenized_test_dataset, id_to_label)
        for i, markers in current_marker_map.items():
            all_predicted_markers[i].extend(markers)

    # Assemble final submission objects
    jsonl_lines = []
    for i in range(len(raw_data)):
        jsonl_obj = {
            "_id": unique_ids[i],
            "conspiracy": conspiracy_keys[i],
            "markers": all_predicted_markers.get(i, [])
        }
        jsonl_lines.append(jsonl_obj)

    # Save and zip the result
    save_and_zip(TEMP_SUBMISSION_FILE, jsonl_lines, FINAL_SUBMISSION_ZIP)

    print("\n Complete")

In [ ]:
#@title 6. Metric Analysis
ground_truth_data = load_data(TEST_FILE)
FINAL_SUBMISSION_ZIP = os.path.join(save_dir, "submission2.zip")

predicted_data = []
try:
    with zipfile.ZipFile(FINAL_SUBMISSION_ZIP, 'r') as zf:
        # Assuming the temporary file name is used inside the zip
        with zf.open(TEMP_SUBMISSION_FILE) as f:
            for line in io.TextIOWrapper(f, encoding='utf-8'):
                predicted_data.append(json.loads(line))
except FileNotFoundError:
    print(f"Error: Submission zip file not found at {FINAL_SUBMISSION_ZIP}")
    sys.exit()

metrics, macro_f1 = compute_final_metrics(ground_truth_data, predicted_data)

metrics_data = []
for marker, m in metrics.items():
    metrics_data.append({
        'Marker_Type': marker,
        'TP': m['TP'], 'FP': m['FP'], 'FN': m['FN'],
        'Precision': m['Precision'],
        'Recall': m['Recall'],
        'F1_Score': m['F1_Score']
    })

df = pd.DataFrame(metrics_data)
df.loc[len(df)] = {
    'Marker_Type': 'MACRO_AVG',
    'TP': '-', 'FP': '-', 'FN': '-',
    'Precision': round(df['Precision'].mean(), 4),
    'Recall': round(df['Recall'].mean(), 4),
    'F1_Score': round(macro_f1, 4)
}

print("\n" + "="*50)
print(f" FINAL SPAN-LEVEL (EXACT MATCH) METRICS")
print(f" Total Markers in Ground Truth: {sum(df.loc[:4, 'TP'] + df.loc[:4, 'FN'])}")
print("="*50 + "\n")
print(df.to_string(index=False))


 FINAL SPAN-LEVEL (EXACT MATCH) METRICS
 Total Markers in Ground Truth: 2677

Marker_Type  TP  FP  FN  Precision  Recall  F1_Score
     Action  29 495 548     0.0553  0.0503    0.0527
      Actor 142 419 648     0.2531  0.1797    0.2102
     Effect  13 406 456     0.0310  0.0277    0.0293
   Evidence  18 333 410     0.0513  0.0421    0.0462
     Victim  52 296 361     0.1494  0.1259    0.1367
  MACRO_AVG   -   -   -     0.1080  0.0851    0.0950
